In [1]:
import torch
import random
import matplotlib.pyplot as plt
import torch.nn.functional as F

!curl -O https://raw.githubusercontent.com/tarikbahar/Turkish_names_and_surnames_dataset/master/Turkce_isimler_listesi.txt
tr_normalization_map=str.maketrans({
    'İ': 'i',
    'I': 'ı',
    'Â': 'a', 'â': 'a',
    'Î': 'i', 'î': 'i',
    'Û': 'u', 'û': 'u',
})

with open('Turkce_isimler_listesi.txt', 'r', encoding='utf-16') as f:
    lines=f.read().splitlines()

processed_words=[]

for line in lines:
    line=line.strip()
    if not line:
        continue
    raw_name=line.split(',')[0].strip()

    clean_name=raw_name.translate(tr_normalization_map).lower()

    if clean_name and clean_name.isalpha():
        processed_words.append(clean_name)

words=sorted(list(set(processed_words)))

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 72050  100 72050    0     0   230k      0 --:--:-- --:--:-- --:--:--  230k


In [2]:
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}
vocab_size = len(itos)


In [3]:
def build_dataset(words):
  block_size = 8
  X, Y = [], []
  for w in words:
     context = [0] * block_size
     for ch in w + '.':
       ix= stoi[ch]
       X.append(context)
       Y.append(ix)
       context = context[1:] + [ix]
  X = torch.tensor(X)
  Y = torch.tensor(Y)
  return X, Y

random.seed(42)
random.shuffle(words)
n1 = int(0.8*len(words))
n2 = int(0.9*len(words))

Xtr, Ytr = build_dataset(words[:n1])
Xdev, Ydev = build_dataset(words[n1:n2])
Xte, Yte = build_dataset(words[n2:])


In [4]:
class Linear:
  def __init__(self, fan_in, fan_out, bias=True):
    self.weight = torch.randn((fan_in, fan_out)) / fan_in**0.5
    self.bias = torch.zeros(fan_out) if bias else None

  def __call__(self, x):
    self.out = x @ self.weight
    if self.bias is not None:
      self.out += self.bias
    return self.out

  def parameters(self):
    return [self.weight] + ([] if self.bias is None else [self.bias])

class BatchNorm1d:
  def __init__(self, dim, eps=1e-5, momentum=0.1):
    self.eps = eps
    self.momentum = momentum
    self.training = True
    self.gamma = torch.ones(dim)
    self.beta = torch.zeros(dim)
    self.running_mean = torch.zeros(dim)
    self.running_var = torch.ones(dim)

  def __call__(self,x):

    if self.training:
      if x.ndim == 2:
        dim = 0
      elif x.ndim == 3:
        dim = (0,1)
      xmean = x.mean(dim, keepdim=True)
      xvar = x.var(dim, keepdim=True)

    else:
      xmean = self.running_mean
      xvar = self.running_var
    xhat = (x-xmean)/torch.sqrt(xvar + self.eps)
    self.out = self.gamma * xhat + self.beta

    if self.training:
      with torch.no_grad():
        self.running_mean = (1-self.momentum)*self.running_mean + self.momentum*xmean
        self.running_var = (1-self.momentum)*self.running_var + self.momentum*xvar

    return self.out

  def parameters(self):
    return([self.gamma, self.beta])

class Tanh:
  def __call__(self, x):
    self.out = torch.tanh(x)
    return self.out
  def parameters(self):
    return[]

class Embedding:
  def __init__(self, num_embeddings, embedding_dim):
    self.weight = torch.randn((num_embeddings, embedding_dim))
  def __call__(self, x):
    self.out = self.weight[x]
    return self.out
  def parameters(self):
    return [self.weight]

class FlattenConsecutive:
  def __init__(self, n):
    self.n = n

  def __call__(self, x):
    B, T, C = x.shape
    x = x.view(B, T//self.n, C*self.n)
    if x.shape[1] == 1:
      x = x.squeeze(1)
    self.out = x
    return self.out
  def parameters(self):
    return []

class Sequential:
  def __init__(self, layers):
    self.layers = layers
  def __call__(self, x):
    for layer in self.layers:
      x = layer(x)
    self.out = x
    return self.out
  def parameters(self):
    return [p for layer in self.layers for p in layer.parameters()]

  def eval(self):
    for layer in self.layers:
      if hasattr(layer, 'training'):
        layer.training = False

  def train(self):
    for layer in self.layers:
      if hasattr(layer, 'training'):
        layer.training = True

In [5]:
torch.manual_seed(42);
block_size = 8
n_embd = 24
n_hidden = 200
C = torch.randn((vocab_size, n_embd))
model = Sequential([
  Embedding(vocab_size, n_embd),
  FlattenConsecutive(2), Linear(n_embd * 2, n_hidden, bias=False), BatchNorm1d(n_hidden), Tanh(),
  FlattenConsecutive(2), Linear(n_hidden*2, n_hidden, bias=False), BatchNorm1d(n_hidden), Tanh(),
  FlattenConsecutive(2), Linear(n_hidden*2, n_hidden, bias=False), BatchNorm1d(n_hidden), Tanh(),
  Linear(n_hidden, vocab_size),
])

with torch.no_grad():
  model.layers[-1].weight *= 0.1

parameters = model.parameters()
print(sum(p.nelement() for p in parameters))
for p in parameters:
  p.requires_grad = True

177550


In [6]:
lossi_flat = []
max_steps = 200000
batch_size = 32
for i in range(max_steps):
  ix = torch.randint(0, Xtr.shape[0], (batch_size,))
  Xb, Yb = Xtr[ix], Ytr[ix]
  logits = model(Xb)
  loss = F.cross_entropy(logits, Yb)
  for p in parameters:
    p.grad = None
  loss.backward()

  lr = 0.1 if i < 100000 else 0.01
  for p in parameters:
    if p.grad is not None:
      p.data += -lr * p.grad

  if i % 10000 == 0:
    print(f'{i:7d}/{max_steps:7d}: {loss.item():.4f}')
  lossi_flat.append(loss.log10().item())

      0/ 200000: 3.4082
  10000/ 200000: 1.1916
  20000/ 200000: 1.5444
  30000/ 200000: 1.1890
  40000/ 200000: 0.8348
  50000/ 200000: 1.3098
  60000/ 200000: 1.3133
  70000/ 200000: 1.0839
  80000/ 200000: 1.0425
  90000/ 200000: 0.7784
 100000/ 200000: 0.9684
 110000/ 200000: 1.3840
 120000/ 200000: 1.2720
 130000/ 200000: 1.2767
 140000/ 200000: 1.1083
 150000/ 200000: 1.3598
 160000/ 200000: 1.0998
 170000/ 200000: 1.1424
 180000/ 200000: 1.2052
 190000/ 200000: 0.9834


In [7]:
@torch.no_grad()
def split_loss(split):
  x, y = {
    'train': (Xtr, Ytr),
    'val': (Xdev, Ydev),
    'test': (Xte, Yte),
  }[split]
  logits = model(x)
  loss = F.cross_entropy(logits, y)
  print(split, loss.item())
split_loss('train')
split_loss('val')

train 1.0720293521881104
val 2.8998661041259766


In [8]:
for _ in range(20):
  out = []
  context = [0] * block_size
  model.eval() # Set model to evaluation mode
  while True:
    logits = model(torch.tensor([context]))
    probs = F.softmax(logits, dim=1)
    ix = torch.multinomial(probs, num_samples=1).item()
    context = context[1:] + [ix]
    out.append(ix)
    if ix == 0:
      break
  model.train() # Set model back to training mode if further training might occur
  print(''.join(itos[i] for i in out))

kınsun.
kutbuldu.
alımga.
töreci.
uruy.
delidumrul.
argun.
akdoğan.
karadoğma.
günçiçek.
koçkar.
bayna.
pusat.
selam.
mehmet.
yelkin.
altınoba.
uluğ.
bilgez.
öküş.




Bağlam-3 MLP      (hafta 4)	                12297	2.20

Bağlam-8 düz MLP     22097                     2.05

Bağlam-8 WaveNet	170897	1.99